In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
from kagglehub import KaggleDatasetAdapter

file_path = "ev_battery_health.csv"
file_dictionary = "data_dictionary .csv"
# Load the latest version
df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "vinay24baghira/ev-battery-health-and-degradation-dataset",
  file_path
)

ddic = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "vinay24baghira/ev-battery-health-and-degradation-dataset",
  file_dictionary
)




c:\scripts\codeInst_projects\Project2\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(df.dtypes)
df.describe(include='all')

battery_id                            str
observation_id                        str
chemistry                             str
manufacturer                          str
vehicle_model                         str
pack_capacity_kwh                 float64
manufacture_date                      str
observation_date                      str
battery_age_days                  float64
cycle_count                       float64
climate_zone                          str
avg_daily_distance_km             float64
avg_charging_power_kw             float64
fast_charge_ratio                 float64
avg_depth_of_discharge_pct        float64
avg_state_of_charge_pct           float64
charge_cycles_per_month           float64
total_energy_throughput_kwh       float64
avg_ambient_temp_c                float64
humidity_pct                      float64
altitude_m                        float64
internal_resistance_mohm          float64
voltage_avg_v                     float64
current_avg_a                     

,battery_id,observation_id,chemistry,manufacturer,vehicle_model,pack_capacity_kwh,manufacture_date,observation_date,battery_age_days,cycle_count,...,peak_temp_during_fast_charge_c,capacity_fade_pct,service_count,last_service_date,bms_firmware_version,recall_flag,state_of_health_pct,remaining_useful_life_cycles,failure_risk_label,degradation_category
count,50000,50000,46541,47539,46485,47698.000000,48090,47784,46998.000000,47731.000000,...,47872.000000,48131.000000,48400.000000,31169,47672,47354.000000,50000.000000,50000.000000,50000,50000
unique,2600,50000,4,8,8,NaN,1515,2106,NaN,NaN,...,NaN,NaN,NaN,2082,5,NaN,NaN,NaN,3,3
top,BATT_01842,OBS_033554,NMC,GigaWatt Labs,Van-E4,NaN,2026-01-24,2026-09-19,NaN,NaN,...,NaN,NaN,NaN,2025-09-23,BMS_v3.0,NaN,NaN,NaN,Low,Normal
freq,37,1,15984,6291,6185,NaN,122,96,NaN,NaN,...,NaN,NaN,NaN,54,9616,NaN,NaN,NaN,42735,46133
mean,NaN,NaN,NaN,NaN,NaN,72.650524,NaN,NaN,561.185455,570.535292,...,26.627565,7.065756,1.428574,NaN,NaN,0.015226,92.935170,1491.181860,NaN,NaN
std,NaN,NaN,NaN,NaN,NaN,14.487910,NaN,NaN,479.605930,599.964295,...,12.914975,7.117441,1.674035,NaN,NaN,0.122451,7.115968,1213.873203,NaN,NaN
min,NaN,NaN,NaN,NaN,NaN,40.000000,NaN,NaN,1.000000,1.000000,...,-10.903885,0.000000,0.000000,NaN,NaN,0.000000,55.000000,0.000000,NaN,NaN
25%,NaN,NaN,NaN,NaN,NaN,62.500000,NaN,NaN,169.000000,137.000000,...,17.800000,1.950000,0.000000,NaN,NaN,0.000000,90.080000,600.000000,NaN,NaN
50%,NaN,NaN,NaN,NaN,NaN,72.600000,NaN,NaN,424.000000,371.000000,...,26.800000,4.800000,1.000000,NaN,NaN,0.000000,95.200000,1142.000000,NaN,NaN
75%,NaN,NaN,NaN,NaN,NaN,82.800000,NaN,NaN,846.000000,801.000000,...,35.600000,9.930000,2.000000,NaN,NaN,0.000000,98.050000,2322.000000,NaN,NaN


In [3]:
# Check for columns with missing data
missing_data = df.isnull().sum()
missing_data = missing_data[missing_data > 0]

if not missing_data.empty:
    print("Columns with missing data and their counts:")
    print(missing_data)
else:
    print("No missing data found in any column.")



Columns with missing data and their counts:
chemistry                          3459
manufacturer                       2461
vehicle_model                      3515
pack_capacity_kwh                  2302
manufacture_date                   1910
observation_date                   2216
battery_age_days                   3002
cycle_count                        2269
climate_zone                       2719
avg_daily_distance_km              2118
avg_charging_power_kw              2140
fast_charge_ratio                  3405
avg_depth_of_discharge_pct         2832
avg_state_of_charge_pct            3138
charge_cycles_per_month            2054
total_energy_throughput_kwh        3370
avg_ambient_temp_c                 3282
humidity_pct                       2737
altitude_m                         2322
internal_resistance_mohm           2104
voltage_avg_v                      2807
current_avg_a                      2487
charge_time_avg_min                2091
peak_temp_during_fast_charge_c     2

In [ ]:
##We can not have these rows in our data because there is not enough time/count data in the reading if all three are missing. 5 rows removed as we can't use those readings.  Forward thinking is time will be important and we can still substitute battery_age_days and cycle count to estimate when a reading happened
columns_to_check = ['observation_date', 'battery_age_days', 'cycle_count']

# Record starting shape
initial_shape = df.shape

# Drop rows where all three columns are null
df = df.dropna(subset=columns_to_check, how='all')

# Print changes
print(f"Original shape: {initial_shape}")
print(f"New shape: {df.shape}")
print(f"Removed rows: {initial_shape[0] - df.shape[0]}")

In [ ]:
def get_non_null_value(battery_id, col_name):
    """
    Looks up a non-null value for a given battery_id and column name from df.
    """
    values = df[df['battery_id'] == battery_id][col_name].dropna()
    if not values.empty:
        return values.iloc[0]
    return np.nan

# Group the DataFrame by 'battery_id' to build an efficient lookup dictionary for each specified column
print("Populating gaps in the dataset for specified columns...")

# Specific columns to fill, these rows related to the battery pack and not the reading. So just for completelness.
columns_to_fill = ["chemistry", "manufacturer", "vehicle_model", "pack_capacity_kwh", "manufacture_date", "climate_zone"]

# Iterate and apply imputation using other rows of the same battery_id
for col in columns_to_fill:
    if col in df.columns:
        # Create a lookup mapping battery_id to its first non-null value for this column
        lookup_map = df.groupby('battery_id')[col].first()

        # Get indices where the column is null
        null_mask = df[col].isnull()

        # Map the battery_id of null rows to the lookup map
        imputed_values = df.loc[null_mask, 'battery_id'].map(lookup_map)

        # Fill the missing values
        df.loc[null_mask, col] = df.loc[null_mask, col].fillna(imputed_values)

# Verify new missing value counts for the selected columns
remaining_missing = df[columns_to_fill].isnull().sum()
print("\nRemaining missing data counts for targeted columns:")
print(remaining_missing)

In [ ]:
# Define columns for each table, there are two tables if we think of the flat table as containing two objects, the battery and its readings.
battery_cols = ["battery_id", "chemistry", "manufacturer", "vehicle_model", "pack_capacity_kwh", "manufacture_date", "climate_zone"]
reading_cols = ["observation_id", "battery_id", "observation_date", "battery_age_days", "cycle_count", "avg_daily_distance_km", "avg_charging_power_kw", "fast_charge_ratio", "avg_depth_of_discharge_pct", "avg_state_of_charge_pct", "charge_cycles_per_month", "total_energy_throughput_kwh", "avg_ambient_temp_c", "humidity_pct", "altitude_m", "internal_resistance_mohm", "voltage_avg_v", "current_avg_a", "charge_time_avg_min", "peak_temp_during_fast_charge_c", "capacity_fade_pct", "service_count", "last_service_date", "bms_firmware_version", "recall_flag", "state_of_health_pct", "remaining_useful_life_cycles", "failure_risk_label", "degradation_category"]

# Create BatteryTable with unique battery_id entries
BatteryTable = df[battery_cols].drop_duplicates(subset=["battery_id"]).reset_index(drop=True)

# Create ReadingTable with unique observation_id entries
# Note: battery_id is included as a foreign key to maintain the relationship
ReadingTable = df[reading_cols].drop_duplicates(subset=["observation_id"]).reset_index(drop=True)

# Display table summaries and previews
print(f"BatteryTable shape: {BatteryTable.shape}")
print(f"ReadingTable shape: {ReadingTable.shape}")

print("\nBatteryTable preview:")
display(BatteryTable.head())

print("\nReadingTable preview:")
display(ReadingTable.head())

In [ ]:
# 1. Ensure observation_date is in datetime64 format ready for sort
ReadingTable['observation_date'] = pd.to_datetime(ReadingTable['observation_date'])

# Print the count of missing cycle_count values BEFORE the operation
initial_missing = ReadingTable['cycle_count'].isnull().sum()
print(f"Missing 'cycle_count' values BEFORE interpolation: {initial_missing}")

# 2. Sort the table chronologically by battery_id and observation_date
ReadingTable = ReadingTable.sort_values(by=['battery_id', 'observation_date']).reset_index(drop=True)

# 3 & 4. Group by battery_id, perform linear interpolation, and apply ffill/bfill for boundary cases
def interpolate_battery_cycles(group):
    # Interpolate over the sorted sequence within each battery group
    group['cycle_count'] = group['cycle_count'].interpolate(method='linear').ffill().bfill()
    return group

# Apply the group-by operation using include_groups=False to avoid DeprecationWarnings
#ReadingTable = ReadingTable.groupby('battery_id', group_keys=False).apply(interpolate_battery_cycles, include_groups=False)

# 5. Print the count of missing cycle_count values AFTER the operation
final_missing = ReadingTable['cycle_count'].isnull().sum()
print(f"Missing 'cycle_count' values AFTER interpolation: {final_missing}")

In [ ]:
display(ReadingTable.head())
display(BatteryTable.head())

In [ ]:
duplicate_count = ReadingTable.duplicated().sum()# No duplicate to sort out
duplicate_count